In [10]:
import os
import json

import numpy as np
import pandas as pd

In [11]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1154, 1194)
# ] # fixed +/- in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1417, 1457)
# ]

# assert len(runs) == 80, len(runs)

original_runs = [
    os.path.join('experiments', str(x)) for x in range(1522, 1562)
]

runs = [
    os.path.join('experiments', str(x)) for x in range(1562, 1582)
]

for run in original_runs:
    hparams = json.load(open(os.path.join(run, 'hparams.json')))
    if hparams['add_bc'] and not hparams['do_masking']:
        continue
    else:
        runs.append(run)

assert len(runs) == 40, len(runs)

In [12]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [13]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [14]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to min. RL2E. (s)  \
Solution        w/ BC                                                
Hopf fibration  False             3.2 ± 0.0             12.3 ± 0.7   
                True              7.4 ± 0.3           108.5 ± 15.1   
Plane waves     False             6.8 ± 0.3             41.7 ± 3.5   
                True              8.1 ± 0.4             52.4 ± 5.1   
Radial waves    False            11.9 ± 0.6            101.7 ± 4.7   
                True             58.9 ± 2.0           499.4 ± 23.1   
Random solution False             6.1 ± 0.2             60.4 ± 2.8   
                True             58.3 ± 3.2            475.5 ± 4.6   

                            Min. RL2E. (%)  
Solution        w/ BC                       
Hopf fibration  False  1.16e-02 ± 4.77e-04  
                True   1.00e-02 ± 1.30e-05  
Plane waves     False  1.09e-02 ± 2.91e-04  
                True   1.03e-02 ± 1.35e-04  
Radial waves    False  1.01e-02 ± 3.67e-05  
                True   2.89e-02 ± 1.80e-03  
Random solution False  1.04e-02 ± 1.21e-04  
                True   1.68e-02 ± 5.52e-04

In [15]:
for column in df.columns:
    if 'time' in column or 'min' in column or 'max' in column or 'mu' in column or 'sigma' in column:
        continue
    try:
        print(column, df[column].unique().tolist())
    except:
        print(column, 'not possible')

input_dim [3]
output_dim [6]
log_freq [100]
verbose [False]
restart_patience [10]
restart_tolerance [0.1]
restarts [False]
model_class ['MaxwellSimple']
norm_inputs [False]
data_from_lattice [False]
n_val [10000]
init ['xavier normal']
noise_scale [0]
inner_lr [0.01]
inner_wd [5e-06]
outer_lr [0.01]
outer_wd [5e-05]
bilevel [False]
symlog [False]
early_stopping [True]
seed [1, 2, 3, 4, 5]
do [0.0]
n_train [2000]
batch_size [1000]
width [10000]
activation ['tanh']
lr [0.05]
wd [5e-05]
beta1 [0.9]
beta2 [0.95]
use_cpu [False]
preload_data [False]
add_bc [True, False]
do_masking [True, False]
soln [4, 3, 2, 1]
scheduler ['cosine']
gain [1.6666666666666667]
c [1.0]
param_count [200000]
soln_name ['Random solution', 'Hopf fibration', 'Radial waves', 'Plane waves']


In [16]:
import torch

In [17]:
x = torch.zeros((10,10))

In [18]:
torch.nn.init.xavier_normal_(x, gain=5)
x.mean()

tensor(0.3096)